<p class="lead">Este suplemento entrega a quien programa todo lo necesario para leer y ejecutar el código R del resto del material de estudio de BGM-411. Es la contraparte en R del suplemento de herramientas de Python, y además cubre los contenidos del primer laboratorio del curso, que el plan oficial plantea en R: vectores, matrices, data frames y listas, y los patrones de indexación de los que dependen todas las unidades posteriores. Si nunca has escrito R, trabaja cada bloque de código (chunk); si ya conoces R, avanza rápido hasta la Sección 6 (el flujo de trabajo de modelamiento).</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Ejecutar R en Google Colab (entorno de ejecución R) y explicar cómo un notebook de R Markdown combina texto, código y salida.</li>
<li>Usar las estructuras de datos fundamentales de R —vectores atómicos, matrices, listas y el <code>data.frame</code>— e indexarlas de cuatro maneras.</li>
<li>Vectorizar un cálculo en lugar de usar un bucle, e interpretar las reglas de reciclaje de R.</li>
<li>Cargar un conjunto de datos desde el paquete <code>mlbench</code>, explorarlo y graficarlo con los gráficos base y con <code>ggplot2</code>.</li>
<li>Llevar a cabo en R un flujo de trabajo completo de aprendizaje supervisado: dividir, entrenar, predecir, evaluar —la plantilla de las Unidades 2–9.</li>
<li>Hacer reproducible un análisis con <code>set.seed</code> y buenos hábitos.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>En Google Colab elige <em>Runtime → Change runtime type → R</em>, o abre el <code>.ipynb</code> (su kernel ya es R). Cada bloque gris es código R; el bloque que está debajo es la salida, exactamente como la produjo R. Los únicos paquetes que se usan en el curso —<code>MASS</code>, <code>class</code>, <code>e1071</code>, <code>rpart</code>, <code>randomForest</code>, <code>cluster</code>, <code>nnet</code>, <code>neuralnet</code>, <code>fastICA</code>, <code>ggplot2</code>, <code>mlbench</code>, <code>pROC</code>— están preinstalados en este entorno; en una sesión nueva de Colab ejecuta <code>install.packages(c("e1071","randomForest","mlbench","pROC"))</code> una vez.</p></div>

## 1. Por qué R, y qué es un notebook

R es un lenguaje construido para los datos: la estadística, el modelamiento y los gráficos forman parte del lenguaje y de su biblioteca estándar, en lugar de estar agregados por fuera.<sup class="c"><a href="#r1">[1]</a></sup> Para este curso R tiene una segunda ventaja: es el lenguaje que usa el plan oficial de laboratorios, de modo que los notebooks sirven también como material de laboratorio; una introducción más amplia y amable es *R for Data Science*.<sup class="c"><a href="#r8">[8]</a></sup> Un **notebook de R Markdown** intercala texto y código R ejecutable; al "tejerlo" (knit), ejecuta cada bloque de código de arriba abajo y fija los resultados junto al código,<sup class="c"><a href="#r2">[2]</a></sup> que es exactamente como se produjeron estas páginas. Por eso, la primera regla de los notebooks es la misma en cualquier lenguaje: **un notebook solo es confiable si se ejecuta de arriba abajo en una sesión nueva**, porque la ejecución fuera de orden oculta errores.



In [ ]:
# Las bibliotecas usadas en el curso. En un entorno R nuevo de Colab, instálalas una vez (ver introducción).
suppressPackageStartupMessages({
  library(ggplot2); library(mlbench)
})
set.seed(0)                      # reproducibilidad: fija todos los sorteos aleatorios posteriores
R.version.string



Lo impreso es la versión de R con la que se construyeron estos apuntes. `set.seed(0)` hace reproducible toda división aleatoria posterior: el hábito más importante de la Sección 7.

## 2. El vocabulario de R: vectores, matrices, listas, data frames

El objeto atómico de R es el **vector**; incluso un solo número es un vector de longitud 1. Los vectores tienen tipo (double, integer, logical, character) y las operaciones están *vectorizadas*: se aplican elemento a elemento, sin bucle.



In [ ]:
x <- c(2, 4, 6, 8, 10)           # vector numérico;  <-  es la asignación
x * 10                            # vectorizado: multiplica cada elemento
x[x > 5]                          # indexación lógica: elementos mayores que 5
c(mean = mean(x), sd = sd(x), n = length(x))   # vector con nombres de resúmenes



Cada línea devuelve un vector: el producto elemento a elemento, el subconjunto seleccionado por una máscara lógica y un vector con nombres de estadísticos de resumen. Cuatro estilos de indexación cubren casi todo lo que harás: por posición positiva (`x[2:3]`), por posición negativa para eliminar (`x[-1]`), por máscara lógica (`x[x > 5]`) y por nombre (`x["mean"]`).



In [ ]:
m <- matrix(1:6, nrow = 2)                    # una matriz de 2x3, llenada por columnas
lst <- list(name = "aspirin", mw = 180.2, ok = TRUE)   # una lista admite tipos mixtos
df <- data.frame(id = 1:3, dose = c(10, 20, 40), responder = c(TRUE, FALSE, TRUE))
cat("matrix dims:", dim(m), "| list$mw:", lst$mw, "\n")
df



Una **matriz** es un vector con forma 2-D (de un solo tipo en todo su contenido); una **lista** es un contenedor ordenado, con nombres y heterogéneo, al que se accede con `$` o `[[ ]]`; y un **`data.frame`** es una lista de columnas de igual longitud: la tabla de R, el objeto que consumen todos los conjuntos de datos y modelos. El `data.frame` es para R lo que el `DataFrame` es para pandas.

## 3. Indexar un data frame



In [ ]:
# Filas por condición, columnas por nombre: df[rows, cols]
df[df$responder, c("id", "dose")]        # solo quienes responden, dos columnas
df$dose[df$dose >= 20]                    # una sola columna y luego una máscara sobre ella
subset(df, dose < 40, select = dose)      # la función auxiliar 'subset', más legible



Las tres celdas muestran el patrón universal `df[rows, columns]`: una condición lógica selecciona filas, un vector de caracteres selecciona columnas y `$` extrae una columna como vector. Dominar este único patrón es el 80 % de la manipulación de datos en R base.

## 4. Vectoriza, no uses bucles



In [ ]:
v <- runif(1e5)                           # 100,000 números aleatorios en [0,1]
t_loop <- system.time({ s <- 0; for (i in seq_along(v)) s <- s + v[i] })["elapsed"]
t_vec  <- system.time({ s2 <- sum(v) })["elapsed"]
cat(sprintf("loop: %.4fs   vectorised sum(): %.5fs   same result: %s\n",
            t_loop, t_vec, isTRUE(all.equal(s, s2))))



La función vectorizada `sum()` da exactamente el mismo resultado, mucho más rápido que el bucle explícito, porque la iteración ocurre en C compilado. La lección se aplica en todo R: prefiere una función vectorizada (`sum`, `rowMeans`, `apply`, `tapply`) a un bucle escrito a mano.<sup class="c"><a href="#r7">[7]</a></sup>

## 5. Un conjunto de datos, y dos maneras de graficarlo

El conjunto de datos recurrente de este curso son los datos de **cáncer de mama de Wisconsin**: 569 aspirados con aguja fina descritos por características de la forma nuclear y etiquetados como benignos o malignos.<sup class="c"><a href="#r3">[3]</a></sup> El paquete `mlbench`<sup class="c"><a href="#r4">[4]</a></sup> incluye una versión, así que no se descarga nada.



In [ ]:
data(BreastCancer, package = "mlbench")
bc <- BreastCancer
bc <- bc[, -1]                                   # elimina la columna Id
bc <- bc[complete.cases(bc), ]                   # elimina las 16 filas con valores faltantes
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))  # ordinal -> numérico
cat("rows x cols:", nrow(bc), "x", ncol(bc), "| classes:\n")
table(bc$Class)



La tabla muestra los conteos por clase (benigno frente a maligno) después de eliminar el identificador y los 16 registros incompletos: el tipo de curación menor que el Suplemento S1 trata en detalle.



In [ ]:
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
boxplot(Cl.thickness ~ Class, data = bc, col = c("#2b5d9b", "#d2691e"),
        xlab = "class", ylab = "clump thickness", main = "Base graphics")
plot(bc$Cell.size, bc$Cell.shape, col = ifelse(bc$Class == "malignant", "#d2691e", "#2b5d9b"),
     pch = 19, cex = 0.5, xlab = "cell size", ylab = "cell shape", main = "malignant (orange)")



<p class="caption"><b>Figura 1.</b> Gráficos de R base: los aspirados malignos (naranja) tienen un grosor del agregado ("clump thickness") sistemáticamente mayor y se concentran en valores altos de tamaño y forma celular: la separabilidad que hace que el problema se pueda aprender a partir de la Unidad 2.</p>



In [ ]:
ggplot(bc, aes(Cell.size, Cell.shape, colour = Class)) +
  geom_jitter(width = 0.3, height = 0.3, alpha = 0.6, size = 1) +
  scale_colour_manual(values = c(benign = "#2b5d9b", malignant = "#d2691e")) +
  labs(title = "The same data with ggplot2", x = "cell size", y = "cell shape") +
  theme_minimal(base_size = 11)



<p class="caption"><b>Figura 2.</b> La misma relación dibujada con <code>ggplot2</code>,<sup class="c"><a href="#r5">[5]</a></sup> que construye un gráfico como una suma de capas (<code>geom_*</code>) y asignaciones estéticas (<code>aes</code>). La dispersión aleatoria (jitter) separa los puntajes enteros superpuestos para que la densidad sea visible. Ambos sistemas gráficos aparecen en el curso; usa el que muestre con más claridad lo que quieres señalar.</p>

## 6. Un flujo de trabajo completo de aprendizaje supervisado

Cada unidad de modelamiento sigue los mismos cinco pasos: dividir los datos, entrenar con la parte de entrenamiento, predecir sobre la parte reservada y evaluar. La celda realiza los cinco con k vecinos más cercanos,<sup class="c"><a href="#r6">[6]</a></sup> el clasificador más simple, porque lo importante es el *flujo de trabajo*, no el modelo.



In [ ]:
library(class)
feat <- bc[, 1:9]; y <- bc$Class
n <- nrow(bc); set.seed(0)
train <- sample(n, round(0.7 * n))               # división 70% entrenamiento / 30% prueba por índice de fila
sc <- scale(feat)                                 # estandarizar: k-NN necesita escalas comparables
pred <- knn(sc[train, ], sc[-train, ], y[train], k = 5)
acc <- mean(pred == y[-train])
cm <- table(truth = y[-train], predicted = pred)
cat(sprintf("test-set accuracy: %.3f  on %d held-out cases\n", acc, length(y) - length(train)))
cm



La matriz de confusión cruza la clase real con la predicción en casos que el modelo nunca vio; la exactitud que aparece sobre ella es la fracción en la diagonal. El Suplemento S1 explica por qué esta estimación con datos reservados —y no la exactitud de entrenamiento— es la honesta, e introduce las métricas más completas (sensibilidad, especificidad, ROC/AUC) que necesitan las unidades médicas.



In [ ]:
ks <- c(1, 3, 5, 9, 15, 25, 41)
accs <- sapply(ks, function(k) mean(knn(sc[train,], sc[-train,], y[train], k = k) == y[-train]))
plot(ks, accs, type = "b", pch = 19, col = "#2b5d9b", xlab = "k (neighbours)",
     ylab = "test accuracy", main = "Choosing k for k-NN")



<p class="caption"><b>Figura 3.</b> Exactitud de prueba en función del número de vecinos k. Un k muy pequeño sobreajusta el ruido de entrenamiento y un k muy grande suaviza en exceso; un valor intermedio es el mejor: el compromiso sesgo–varianza que el Suplemento S1 formaliza.</p>

## 7. Reproducibilidad y buenos hábitos



In [ ]:
set.seed(42); a <- sample(5)
set.seed(42); b <- sample(5)
cat("same seed -> identical draw:", identical(a, b), "\n")
sessionInfo()$R.version$version.string



Fijar la semilla hace que un análisis aleatorio sea repetible hasta el sorteo exacto, como muestran los vectores idénticos. Los hábitos que hacen confiable un análisis en R: fija una semilla una sola vez, al inicio; nunca dependas de objetos creados fuera de orden; mantén los datos crudos como solo lectura; carga los paquetes al inicio; y registra la sesión (`sessionInfo()`) para que quien lea sepa las versiones. Esta es la forma en R de las reglas de reproducibilidad a las que el curso vuelve en la Unidad 9.

## 8. Resumen

- Un notebook debe ejecutarse de arriba abajo en una sesión nueva; `set.seed` lo hace reproducible.
- Los objetos de R: vectores con tipo (las operaciones se vectorizan), matrices (un solo tipo, 2-D), listas (mixtas, con nombres), data frames (la tabla que consume todo modelo).
- Indexa con `df[rows, cols]`: posiciones, negativos, máscaras lógicas o nombres.
- Prefiere funciones vectorizadas a los bucles.
- `mlbench` proporciona los datos de cáncer de mama; tanto los gráficos base como `ggplot2` permiten graficarlos.
- El flujo de trabajo supervisado es siempre dividir → entrenar → predecir → evaluar con datos reservados; la matriz de confusión y la exactitud resumen un clasificador.

## 9. Autoevaluación

<details><summary>1. ¿Por qué un notebook debe ejecutarse de arriba abajo en una sesión nueva?</summary>Porque ejecutar los bloques de código fuera de orden puede dejar variables en un estado que el código no reproduce en una ejecución limpia, lo que oculta errores y vuelve irreproducibles los resultados. Tejer el documento (knit), como aquí, obliga a una ejecución limpia de arriba abajo.</details>
<details><summary>2. ¿Qué devuelve <code>x[x > 5]</code> y cuál es el mecanismo?</summary>Los elementos de <code>x</code> mayores que 5. <code>x > 5</code> construye un vector lógico; indexar con él conserva las posiciones <code>TRUE</code> (indexación lógica/booleana).</details>
<details><summary>3. ¿Qué diferencia hay entre una matriz, una lista y un data frame?</summary>Una matriz es un arreglo 2-D de un solo tipo; una lista es un contenedor ordenado, con nombres y de tipos mixtos; un data frame es una lista de columnas de igual longitud (posiblemente de tipos distintos): la tabla de R.</details>
<details><summary>4. ¿Por qué <code>sum(v)</code> es mucho más rápido que un bucle <code>for</code> que suma <code>v</code>?</summary>Las funciones vectorizadas realizan su iteración en C compilado, lo que evita la sobrecarga del intérprete de un bucle escrito en R.</details>
<details><summary>5. En <code>df[df$responder, c("id","dose")]</code>, ¿qué selecciona las filas y qué selecciona las columnas?</summary>El vector lógico <code>df$responder</code> selecciona las filas; el vector de caracteres <code>c("id","dose")</code> selecciona las columnas, según el patrón <code>df[rows, cols]</code>.</details>
<details><summary>6. ¿Por qué estandarizar las características antes de aplicar k-NN?</summary>k-NN usa la distancia euclidiana, que queda dominada por las características con escalas numéricas más grandes; estandarizar (<code>scale</code>) pone las características en escalas comparables para que todas contribuyan.</details>
<details><summary>7. ¿Qué muestra la matriz de confusión que un único número de exactitud oculta?</summary>El desglose de los errores por tipo —cuántos casos malignos no se detectaron (falsos negativos) frente a cuántos benignos se clasificaron como malignos (falsos positivos)—, lo que importa porque ambos errores tienen costos clínicos distintos.</details>
<details><summary>8. ¿Por qué evaluar con <code>y[-train]</code> y no con las filas de entrenamiento?</summary>La exactitud de entrenamiento tiene un sesgo optimista porque el modelo ya vio esos casos; el conjunto reservado estima el rendimiento con datos no vistos (Suplemento S1).</details>
<details><summary>9. ¿Cuál es el efecto de k en k-NN, según la Figura 3?</summary>Un k pequeño sobreajusta (alta varianza); un k grande suaviza en exceso (alto sesgo); un k intermedio generaliza mejor.</details>
<details><summary>10. ¿Qué dos cosas garantiza <code>set.seed(42)</code> antes de <code>sample()</code>?</summary>Que el sorteo aleatorio sea reproducible —la misma semilla produce una secuencia idéntica—, de modo que la división y cualquier modelo estocástico puedan reproducirse exactamente.</details>
<details><summary>11. ¿Cómo eliminas una columna por nombre y cómo por posición?</summary>Por posición, con un índice negativo (<code>bc[, -1]</code>); por nombre, seleccionando el complemento (<code>bc[, setdiff(names(bc), "Id")]</code>).</details>
<details><summary>12. ¿Qué debe ir al comienzo mismo de un notebook de R reproducible?</summary>La carga de bibliotecas y un único <code>set.seed</code>, antes de cualquier análisis, de modo que las dependencias y la aleatoriedad queden fijadas desde el inicio.</details>

## Glosario

<dl class="gloss">
<dt>Vector (vector)</dt><dd>El objeto atómico de R: una secuencia 1-D con tipo; las operaciones se aplican elemento a elemento (vectorizadas).</dd>
<dt>Matriz (matrix)</dt><dd>Un vector con forma 2-D; todos los elementos comparten un mismo tipo.</dd>
<dt>Lista (list)</dt><dd>Un contenedor ordenado y con nombres de elementos de tipos mixtos, al que se accede con <code>$</code> o <code>[[ ]]</code>.</dd>
<dt>data.frame</dt><dd>Una lista de columnas de igual longitud; la tabla de R y la entrada de todo modelo.</dd>
<dt>Vectorización (vectorisation)</dt><dd>Aplicar una operación a un vector completo de una sola vez, en código compilado, en lugar de usar un bucle.</dd>
<dt>Indexación lógica (logical indexing)</dt><dd>Seleccionar elementos o filas con una máscara de <code>TRUE</code>/<code>FALSE</code>.</dd>
<dt>Matriz de confusión (confusion matrix)</dt><dd>Una tabla cruzada de las clases reales frente a las predichas en un conjunto de prueba.</dd>
<dt>Semilla (seed)</dt><dd>El estado inicial del generador de números aleatorios; fijarla hace reproducibles los resultados aleatorios.</dd>
<dt>R Markdown</dt><dd>Un formato de documento que intercala texto y bloques de código R y que se teje (knit) a HTML con la salida ya ejecutada.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r2">Xie Y, Allaire JJ, Grolemund G. <em>R Markdown: The Definitive Guide</em>. Chapman &amp; Hall/CRC; 2018. <a href="https://doi.org/10.1201/9781138359444">doi:10.1201/9781138359444</a></li>
<li id="r3">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r4">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r5">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
<li id="r6">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r7">Wickham H. <em>Advanced R</em>. 2nd ed. Chapman &amp; Hall/CRC; 2019. <a href="https://doi.org/10.1201/9781351201315">doi:10.1201/9781351201315</a></li>
<li id="r8">Wickham H, Grolemund G. <em>R for Data Science</em>. O'Reilly; 2017. <a href="https://r4ds.had.co.nz/">https://r4ds.had.co.nz/</a> (de libre acceso; sin DOI)</li>
</ol>

<h4>Libros de texto y recursos</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 —el texto complementario en R para todo el curso; PDF gratuito en statlearning.com.</li>
<li>Venables &amp; Ripley, <em>Modern Applied Statistics with S</em> (MASS) —la referencia para las funciones de modelamiento de R que se usan aquí.</li>
<li>Xie, <em>R Markdown: The Definitive Guide</em> y las hojas de referencia (cheatsheets) de RStudio/Posit —para el formato de notebook.</li>
</ul>
